In [2]:
import sys
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
sys.path.append(str(Path.cwd().parent / "src"))
import csv
import numpy
import utils.data_loader as dl
import utils.data_summaries as ds
import utils.standardize_columns as sc

In [3]:
print(dir(dl))
print(dir(ds))
print(dir(sc))

['Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'csv', 'get_sources', 'pd', 'pull_data', 'read_files']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'check_duplicates', 'column_presence', 'compare_columns', 'full_summary', 'pd', 'quick_summary', 'search_df']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_resolve', 'col_to_bool', 'col_to_cat', 'col_to_date', 'col_to_float', 'col_to_int', 'col_to_snake', 'col_to_str', 'pd', 're']


In [4]:
#Read available datasets available for data source
path = "C:\\Users\\1\\Desktop\\project\\data\\cleaned_data"
DataSources = dl.get_sources(path)
print(DataSources)

[WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/climate'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/neighbourhoods'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/watermain_breaks'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/watermains'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/weather')]


## All sheets

### To-be-combined Metric tables
Dwellings and Family data collected for 2011 is stored in the households data, we can combine Dwellings and Family into households and have a households metric data in the structure of 

|Variable|Ward|Year|Value|Category|


Reconvert the transformed wide table back to long and union them

In [5]:
source = DataSources[3]
readable = dl.read_files(source)
readable

CSV
  dwellings_long.csv
  education_long.csv
  ethnic_long.csv
  families_long.csv
  households_long.csv
  income_long.csv
  labour_long.csv
  population_long.csv


[WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/dwellings_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/education_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/ethnic_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/families_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/households_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/income_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/labour_long.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/ward_census/population_long.csv')]

In [6]:
data = dl.pull_data(source)
print(f"number of datasets from source: {len(data)}")
print(f"names of remaining datasets: {data.keys()}")

CSV
  dwellings_long.csv
  education_long.csv
  ethnic_long.csv
  families_long.csv
  households_long.csv
  income_long.csv
  labour_long.csv
  population_long.csv
number of datasets from source: 8
names of remaining datasets: dict_keys(['dwellings_long', 'education_long', 'ethnic_long', 'families_long', 'households_long', 'income_long', 'labour_long', 'population_long'])


### Drop Totals

In [7]:
def drop_totals(df):
    df = df[~df["variable"].str.contains("total", case=False, na=False)] #drop variable containing string "total"
    df = df[~df["geography"].str.contains("toronto", case=False, na=False)].reset_index(drop=True) # drop geography = Toronto
    return df

## Dwellings Transformation 1/8

In [108]:
dwelling = data['dwellings_long']
dwelling[dwelling['geography']=='Ward 1']

,variable,geography,value,year,category
25,Total - Occupied private dwellings by structur...,Ward 1,38135,2021,dwellings
26,Single-detached house,Ward 1,11270,2021,dwellings
27,Semi-detached house,Ward 1,1425,2021,dwellings
28,Row house,Ward 1,3645,2021,dwellings
29,Apartment or flat in a duplex,Ward 1,3790,2021,dwellings
30,Apartment in a building that has fewer than fi...,Ward 1,2015,2021,dwellings
31,Apartment in a building that has five or more ...,Ward 1,15980,2021,dwellings
32,Other single-attached house,Ward 1,10,2021,dwellings
33,Movable dwelling,Ward 1,0,2021,dwellings
34,Total - Tenure (includes band housing),Ward 1,38130,2021,dwellings


In [8]:
#Clean dwellings census data
dwelling = data['dwellings_long']
print(ds.quick_summary(dwelling))
dwelling = dwelling[~dwelling["variable"].str.contains("total", case=False, na=False)] #drop variable containing string "total"
dwelling = dwelling[~dwelling["geography"].str.contains("toronto", case=False, na=False)].reset_index(drop=True) # drop geography = Toronto
dwelling

            dtype  col_index  missing_pct  n_unique
variable   object          0          0.0        24
geography  object          1          0.0        26
value       int64          2          0.0       985
year        int64          3          0.0         2
category   object          4          0.0         1


,variable,geography,value,year,category
0,Single-detached house,Ward 1,11270,2021,dwellings
1,Semi-detached house,Ward 1,1425,2021,dwellings
2,Row house,Ward 1,3645,2021,dwellings
3,Apartment or flat in a duplex,Ward 1,3790,2021,dwellings
4,Apartment in a building that has fewer than fi...,Ward 1,2015,2021,dwellings
...,...,...,...,...,...
1070,1981 to 1990,Ward 25,8940,2016,dwellings
1071,1991 to 2000,Ward 25,5050,2016,dwellings
1072,2001 to 2005,Ward 25,3205,2016,dwellings
1073,2006 to 2010,Ward 25,1665,2016,dwellings


In [9]:
dwelling['variable'].value_counts()

variable
Not part of a condominium                                   100
Part of a condominium                                       100
Semi-detached house                                          50
2006 to 2010                                                 50
2001 to 2005                                                 50
1991 to 2000                                                 50
1981 to 1990                                                 50
1961 to 1980                                                 50
1960 or before                                               50
Rented                                                       50
Single-detached house                                        50
Owned                                                        50
Movable dwelling                                             50
Other single-attached house                                  50
Apartment in a building that has five or more storeys        50
Apartment in a building that ha

In [10]:
totals = ds.search_df(dwelling, "total", None,"variable")
totals["variable"]

Series([], Name: variable, dtype: object)

In [11]:
dwelling['year'].value_counts()

year
2021    550
2016    525
Name: count, dtype: int64

### Mapping and transform

In [12]:
import pandas as pd


def transform_dwellings_reduced(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Clean string columns just in case
    df["variable"] = df["variable"].astype(str).str.strip()

    # ----------------------------
    # 1. Exhaustive mappings
    # ----------------------------
    type_map = {
        "Single-detached house": "detached",
        "Semi-detached house": "attached/other",
        "Row house": "attached/other",
        "Other single-attached house": "attached/other",
        "Apartment or flat in a duplex": "apartment",
        "Apartment in a building that has fewer than five storeys": "apartment",
        "Apartment in a building that has five or more storeys": "apartment",
        "Movable dwelling": "attached/other"   # very negligible, fold into attached instead of other
    }

    condo_map = {
        "Part of a condominium": "condo",
        "Not part of a condominium": "non_condo",
    }

    ownership_map = {
        "Owned": "owned",
        "Rented": "rented",
    }

    age_map = {
        "1960 or before": "pre_1980",
        "1961 to 1980": "pre_1980",
    
        "1981 to 1990": "1980_2000",
        "1991 to 2000": "1980_2000",
    
        "2001 to 2005": "2000_2010",
        "2006 to 2010": "2000_2010",
    
        "2011 to 2015": "2010_2020",
        "2011 to 2016": "2010_2020",  # dataset variant
        "2016 to 2021": "2010_2020",
    
        # future-proof (in case newer data appears)
        "2021 to 2026": "post_2020",
        "2022 to 2026": "post_2020",
    }

    # ----------------------------
    # 2. Assign reduced groups
    # ----------------------------
    df["type_group"] = df["variable"].map(type_map)
    df["condo_group"] = df["variable"].map(condo_map)
    df["ownership_group"] = df["variable"].map(ownership_map)
    df["age_group"] = df["variable"].map(age_map)

    # ----------------------------
    # 3. Validate coverage
    # ----------------------------
    covered = (
        df["type_group"].notna()
        | df["condo_group"].notna()
        | df["ownership_group"].notna()
        | df["age_group"].notna()
    )

    unmapped = sorted(df.loc[~covered, "variable"].dropna().unique())
    if unmapped:
        raise ValueError(f"Unmapped dwelling variables found: {unmapped}")

    # ----------------------------
    # 4. Helper to compute subgroup %
    # ----------------------------
    def compute_pct(subset: pd.DataFrame, group_col: str) -> pd.DataFrame:
        grouped = (
            subset.groupby(["geography", "year", group_col], as_index=False)["value"]
            .sum()
        )

        grouped["denominator"] = grouped.groupby(["geography", "year"])["value"].transform("sum")
        grouped["pct"] = grouped["value"] / grouped["denominator"]

        wide = (
            grouped.pivot(index=["geography", "year"], columns=group_col, values="pct")
            .reset_index()
        )
        wide.columns.name = None
        return wide

    # ----------------------------
    # 5. Build reduced blocks
    # ----------------------------
    type_df = compute_pct(df[df["type_group"].notna()], "type_group")
    condo_df = compute_pct(df[df["condo_group"].notna()], "condo_group")
    ownership_df = compute_pct(df[df["ownership_group"].notna()], "ownership_group")
    age_df = compute_pct(df[df["age_group"].notna()], "age_group")

    # ----------------------------
    # 6. Merge blocks
    # ----------------------------
    final = type_df.merge(condo_df, on=["geography", "year"], how="outer")
    final = final.merge(ownership_df, on=["geography", "year"], how="outer")
    final = final.merge(age_df, on=["geography", "year"], how="outer")

    # ----------------------------
    # 7. Rename final columns
    # ----------------------------
    final = final.rename(columns={
        "detached": "pct_detached",
        "attached/other": "pct_attached/other",
        "apartment": "pct_apartment",
        "condo": "pct_condo",
        "non_condo": "pct_non_condo",
        "owned": "pct_owned",
        "rented": "pct_rented",
        "pre_1980": "pct_pre_1980",
        "1980_2000": "pct_1980_2000",
        "2000_2010": "pct_2000_2010",
        "2010_2020": "pct_2010_2020",
        "post_2020": "pct_post_2020",
    })

    # Optional: keep only the most useful reduced features
    keep_cols = [
        "geography",
        "year",
        "pct_detached",
        "pct_attached/other",
        "pct_apartment",
        "pct_condo",
        "pct_non_condo",
        "pct_pre_1980",
        "pct_1980_2000",
        "pct_2000_2010",
        "pct_2010_2020",
        "pct_post_2020",
    ]
    existing_cols = [c for c in keep_cols if c in final.columns]
    final = final[existing_cols]

    return final

In [13]:
dwellings_trans = transform_dwellings_reduced(dwelling)
dwellings_trans

,geography,year,pct_detached,pct_attached/other,pct_apartment,pct_condo,pct_non_condo,pct_pre_1980,pct_1980_2000,pct_2000_2010,pct_2010_2020
0,Ward 1,2016,0.296252,0.131169,0.572579,0.222853,0.777147,0.667634,0.261776,0.052909,0.017680
1,Ward 1,2021,0.295529,0.133211,0.571260,0.233805,0.766195,0.638772,0.267707,0.058499,0.035021
2,Ward 10,2016,0.007581,0.049315,0.943103,0.793405,0.206595,0.163085,0.172873,0.347733,0.316309
3,Ward 10,2021,0.006193,0.041246,0.952561,0.833148,0.166852,0.132541,0.146166,0.271832,0.449461
4,Ward 11,2016,0.082699,0.118239,0.799063,0.311432,0.688568,0.650625,0.147794,0.111187,0.090394
5,Ward 11,2021,0.078124,0.116157,0.805719,0.382020,0.617980,0.573031,0.139893,0.107653,0.179422
6,Ward 12,2016,0.166729,0.072540,0.760731,0.186018,0.813982,0.721327,0.147770,0.093047,0.037856
7,Ward 12,2021,0.152284,0.064441,0.783275,0.274847,0.725153,0.616384,0.139059,0.080802,0.163755
8,Ward 13,2016,0.004016,0.051102,0.944881,0.424763,0.575237,0.464536,0.233123,0.170227,0.132114
9,Ward 13,2021,0.003625,0.045824,0.950551,0.528132,0.471868,0.366154,0.197578,0.132323,0.303944


In [14]:
def validate_dwellings_output(
    df: pd.DataFrame,
    tol: float = 1e-3,
    verbose: bool = True
) -> dict:
    """
    Validate that percentage groups sum to ~1.

    Returns dict with summary + problematic rows.
    """

    results = {}

    groups = {
        "type": ["pct_detached", "pct_attached/other", "pct_apartment"],
        "age": [
            "pct_pre_1980",
            "pct_1980_2000",
            "pct_2000_2010",
            "pct_2010_2020",
            "pct_post_2020",
        ],
        "condo": ["pct_condo", "pct_non_condo"],
    }

    for group_name, cols in groups.items():
        existing_cols = [c for c in cols if c in df.columns]

        if not existing_cols:
            continue

        df[f"{group_name}_sum"] = df[existing_cols].sum(axis=1)

        bad_mask = ~df[f"{group_name}_sum"].between(1 - tol, 1 + tol)

        results[group_name] = {
            "columns": existing_cols,
            "mean_sum": df[f"{group_name}_sum"].mean(),
            "min_sum": df[f"{group_name}_sum"].min(),
            "max_sum": df[f"{group_name}_sum"].max(),
            "bad_rows_count": bad_mask.sum(),
            "bad_rows": df.loc[bad_mask, ["geography", "year", f"{group_name}_sum"]],
        }

        if verbose:
            print(f"\n--- {group_name.upper()} CHECK ---")
            print(f"Columns: {existing_cols}")
            print(f"Mean: {results[group_name]['mean_sum']:.6f}")
            print(f"Min:  {results[group_name]['min_sum']:.6f}")
            print(f"Max:  {results[group_name]['max_sum']:.6f}")
            print(f"Bad rows: {results[group_name]['bad_rows_count']}")

    return results

In [15]:
dwellings_trans.columns

Index(['geography', 'year', 'pct_detached', 'pct_attached/other',
       'pct_apartment', 'pct_condo', 'pct_non_condo', 'pct_pre_1980',
       'pct_1980_2000', 'pct_2000_2010', 'pct_2010_2020'],
      dtype='object')

## Education Transformation 2/8

Hierarchy looks as thus: 
Total - Highest certificate, diploma or degree
│
├── No certificate, diploma or degree
│
├── High (secondary) school diploma or equivalency certificate
│
└── Postsecondary certificate, diploma or degree
    │
    ├── Postsecondary certificate or diploma below bachelor level
    │   │
    │   ├── Apprenticeship or trades certificate or diploma
    │   │   ├── Apprenticeship certificate
    │   │   └── Non-apprenticeship trades certificate or diploma
    │   │
    │   ├── College, CEGEP or other non-university certificate or diploma
    │   │
    │   └── University certificate or diploma below bachelor level
    │
    └── Bachelor's degree or higher
        │
        ├── Bachelor's degree
        │
        ├── University certificate or diploma above bachelor level
        │
        ├── Degree in medicine, dentistry, veterinary medicine or optometry
        │
        ├── Master's degree
        │
        └── Earned doctorate


In [120]:
# ----------------------------
# EDUCATION MAPPING (AI-READY)
# ----------------------------
# Goal:
# Convert detailed census education categories into a flat, mutually exclusive
# distribution that sums to 1 and represents an ordinal education gradient.
#
# IMPORTANT RULES:
# - Only use LEAF categories (no aggregates)
# - Do NOT include totals or rollups (they cause double counting)
# - All groups must be mutually exclusive
# - Groups should represent increasing education level (ordinal structure)
#
# Final interpretation for models:
# pct_no_highschool         → lowest attainment
# pct_highschool            → basic attainment
# pct_postsecondary_non_uni → trades / college
# pct_university_plus       → highest attainment
#
# These features:
# - sum to ~1 within each geography/year
# - encode a monotonic socioeconomic gradient
# - are safe for aggregation and modeling

edu_map = {
    # --- Lowest education ---
    "No certificate, diploma or degree": "no_highschool",

    # --- High school ---
    "High (secondary) school diploma or equivalency certificate": "highschool",
    "Secondary (high) school diploma or equivalency certificate": "highschool",

    # --- Postsecondary (NON-university) ---
    # NOTE: only include leaf categories, NOT aggregates like:
    # "Postsecondary certificate, diploma or degree"
    # "Postsecondary certificate or diploma below bachelor level"
    # "Apprenticeship or trades certificate or diploma"
    "Apprenticeship certificate": "postsecondary_non_uni",
    "Non-apprenticeship trades certificate or diploma": "postsecondary_non_uni",
    "Trades certificate or diploma other than Certificate of Apprenticeship or Certificate of Qualification": "postsecondary_non_uni",
    "Certificate of Apprenticeship or Certificate of Qualification": "postsecondary_non_uni",
    "College, CEGEP or other non-university certificate or diploma": "postsecondary_non_uni",
    "University certificate or diploma below bachelor level": "postsecondary_non_uni",

    # --- University and above ---
    # NOTE: exclude aggregate rows like:
    # "Bachelor’s degree or higher"
    # "University certificate, diploma or degree at bachelor level or above"
    "Bachelor's degree": "university_plus",
    "University certificate or diploma above bachelor level": "university_plus",
    "Master's degree": "university_plus",
    "Earned doctorate": "university_plus",
    "Degree in medicine, dentistry, veterinary medicine or optometry": "university_plus",
}

In [109]:
edu = data['education_long']
edu[edu['geography']=='Ward 1']

,variable,geography,value,year,category
16,"Total - Highest certificate, diploma or degree...",Ward 1,96620,2021,education
17,"No certificate, diploma or degree",Ward 1,21870,2021,education
18,High (secondary) school diploma or equivalency...,Ward 1,28780,2021,education
19,"Postsecondary certificate, diploma or degree",Ward 1,45965,2021,education
20,Postsecondary certificate or diploma below bac...,Ward 1,23845,2021,education
21,Apprenticeship or trades certificate or diploma,Ward 1,4400,2021,education
22,Non-apprenticeship trades certificate or diploma,Ward 1,2580,2021,education
23,Apprenticeship certificate,Ward 1,1820,2021,education
24,"College, CEGEP or other non-university certifi...",Ward 1,16295,2021,education
25,University certificate or diploma below bachel...,Ward 1,3145,2021,education


In [16]:
#Clean dwellings census data
edu = data['education_long']
print(ds.quick_summary(edu))
edu = edu[~edu["variable"].str.contains("total", case=False, na=False)] #drop variable containing string "total"
edu = edu[~edu["geography"].str.contains("toronto", case=False, na=False)].reset_index(drop=True) # drop geography = Toronto
edu

            dtype  col_index  missing_pct  n_unique
variable   object          0          0.0        20
geography  object          1          0.0        26
value       int64          2          0.0       717
year        int64          3          0.0         2
category   object          4          0.0         1


,variable,geography,value,year,category
0,"No certificate, diploma or degree",Ward 1,21870,2021,education
1,High (secondary) school diploma or equivalency...,Ward 1,28780,2021,education
2,"Postsecondary certificate, diploma or degree",Ward 1,45965,2021,education
3,Postsecondary certificate or diploma below bac...,Ward 1,23845,2021,education
4,Apprenticeship or trades certificate or diploma,Ward 1,4400,2021,education
...,...,...,...,...,...
720,Bachelor's degree,Ward 25,15120,2016,education
721,University certificate or diploma above bachel...,Ward 25,1280,2016,education
722,"Degree in medicine, dentistry, veterinary medi...",Ward 25,420,2016,education
723,Master's degree,Ward 25,3670,2016,education


In [17]:
edu['variable'].value_counts() #we will drop aggregates and keep leaf variables, re-aggregate manually after

variable
No certificate, diploma or degree                                                                         50
Master's degree                                                                                           50
Postsecondary certificate, diploma or degree                                                              50
Apprenticeship or trades certificate or diploma                                                           50
College, CEGEP or other non-university certificate or diploma                                             50
University certificate or diploma below bachelor level                                                    50
Earned doctorate                                                                                          50
Bachelor's degree                                                                                         50
University certificate or diploma above bachelor level                                                    50
Degree in 

In [111]:
edu_leaf = edu[
    ~edu["variable"].str.contains("Postsecondary certificate, diploma or degree", case=False, na=False)
]

edu_leaf = edu_leaf[
    ~edu_leaf["variable"].str.contains("Bachelor’s degree or higher", case=False, na=False)
]

edu_leaf = edu_leaf[
    ~edu_leaf["variable"].str.contains("bachelor level or above", case=False, na=False)
].reset_index(drop=True)
edu_leaf

,variable,geography,value,year,category
0,"Total - Highest certificate, diploma or degree...",Toronto,2377950,2021,education
1,"No certificate, diploma or degree",Toronto,339500,2021,education
2,High (secondary) school diploma or equivalency...,Toronto,555590,2021,education
3,Postsecondary certificate or diploma below bac...,Toronto,506235,2021,education
4,Apprenticeship or trades certificate or diploma,Toronto,80930,2021,education
...,...,...,...,...,...
697,Bachelor's degree,Ward 25,15120,2016,education
698,University certificate or diploma above bachel...,Ward 25,1280,2016,education
699,"Degree in medicine, dentistry, veterinary medi...",Ward 25,420,2016,education
700,Master's degree,Ward 25,3670,2016,education


In [19]:
edu_leaf['variable'].value_counts()

variable
No certificate, diploma or degree                                                                         50
University certificate or diploma below bachelor level                                                    50
Earned doctorate                                                                                          50
Master's degree                                                                                           50
Degree in medicine, dentistry, veterinary medicine or optometry                                           50
University certificate or diploma above bachelor level                                                    50
Bachelor's degree                                                                                         50
College, CEGEP or other non-university certificate or diploma                                             50
Apprenticeship or trades certificate or diploma                                                           50
Bachelor’s

### Mapping and transform

In [118]:
def transform_education(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["variable"] = df["variable"].astype(str).str.strip()

    # mapping
    edu_map = {
        "No certificate, diploma or degree": "no_highschool",
        "High (secondary) school diploma or equivalency certificate": "highschool",
        "Secondary (high) school diploma or equivalency certificate": "highschool",

        #"Apprenticeship or trades certificate or diploma": "postsecondary_non_uni",
        "Apprenticeship certificate": "postsecondary_non_uni",
        "Non-apprenticeship trades certificate or diploma": "postsecondary_non_uni",
        "Trades certificate or diploma other than Certificate of Apprenticeship or Certificate of Qualification": "postsecondary_non_uni",
        "Certificate of Apprenticeship or Certificate of Qualification": "postsecondary_non_uni",
        "College, CEGEP or other non-university certificate or diploma": "postsecondary_non_uni",
        "University certificate or diploma below bachelor level": "postsecondary_non_uni",
        #"Postsecondary certificate or diploma below bachelor level": "postsecondary_non_uni",

        "Bachelor's degree": "university_plus",
        "University certificate or diploma above bachelor level": "university_plus",
        #"University certificate, diploma or degree at bachelor level or above": "university_plus",
        "Master's degree": "university_plus",
        "Earned doctorate": "university_plus",
        "Degree in medicine, dentistry, veterinary medicine or optometry": "university_plus",
    }

    # map groups
    df["edu_group"] = df["variable"].map(edu_map)

    # filter valid rows only (drops aggregates like "Postsecondary certificate, diploma or degree")
    df = df[df["edu_group"].notna()].copy()

    # aggregate
    grouped = (
        df.groupby(["geography", "year", "edu_group"], as_index=False)["value"]
        .sum()
    )

    # compute %
    grouped["total"] = grouped.groupby(["geography", "year"])["value"].transform("sum")
    grouped["pct"] = grouped["value"] / grouped["total"]

    # pivot wide
    wide = (
        grouped.pivot(index=["geography", "year"], columns="edu_group", values="pct")
        .reset_index()
    )

    wide.columns.name = None

    # rename final columns
    wide = wide.rename(columns={
        "no_highschool": "pct_no_highschool",
        "highschool": "pct_highschool",
        "postsecondary_non_uni": "pct_postsecondary_non_uni",
        "university_plus": "pct_university_plus",
    })

    return wide

In [119]:
max_edu = transform_education(edu_leaf)
max_edu[max_edu['geography'] == 'Ward 1']

,geography,year,pct_highschool,pct_no_highschool,pct_postsecondary_non_uni,pct_university_plus
2,Ward 1,2016,0.306502,0.249095,0.250879,0.193525
3,Ward 1,2021,0.297868,0.226351,0.246740,0.229042


## Ethnic Transformation 3/8
ethnic_long

In [125]:
ethnic = data['ethnic_long']
ethnic [ ethnic['geography']=='Ward 1']

,variable,geography,value,year,category
266,Total - Ethnic or cultural origin for the popu...,Ward 1,115115,2021,ethnic
267,Canadian,Ward 1,6335,2021,ethnic
268,English,Ward 1,3330,2021,ethnic
269,Irish,Ward 1,2945,2021,ethnic
270,Scottish,Ward 1,2640,2021,ethnic
...,...,...,...,...,...
7499,Korean,Ward 1,480,2016,ethnic
7500,Japanese,Ward 1,145,2016,ethnic
7501,"Visible minority, n.i.e.",Ward 1,4115,2016,ethnic
7502,Multiple visible minorities,Ward 1,2355,2016,ethnic


In [127]:
ethnic.variable.unique()

array(['Total - Ethnic or cultural origin for the population in private households - 25% sample data',
       'Canadian', 'English', 'Irish', 'Scottish', 'French, n.o.s.',
       'German', 'Chinese', 'Italian', 'Indian (India)', 'Ukrainian',
       'Dutch', 'Polish', 'Québécois', 'British Isles, n.o.s.',
       'Filipino', 'French Canadian', 'Caucasian (White), n.o.s.',
       'First Nations (North American Indian), n.o.s.', 'Métis',
       'European, n.o.s.', 'Russian', 'Norwegian', 'Welsh', 'Portuguese',
       'American', 'Spanish', 'Swedish', 'Hungarian', 'Acadian',
       'Pakistani', 'African, n.o.s.', 'Jewish', 'Punjabi', 'Vietnamese',
       'Arab, n.o.s.', 'Greek', 'Jamaican', 'Asian, n.o.s.',
       'Cree, n.o.s.', 'Korean', 'Romanian', 'Lebanese', 'Iranian',
       'Christian, n.i.e.', 'Danish', 'North American Indigenous, n.o.s.',
       'Sikh', 'Austrian', 'Belgian', 'Haitian', 'Hindu', 'Mexican',
       'Mennonite', 'Swiss', 'Finnish', 'Sri Lankan', 'Croatian',
       'Ja

In [128]:
ethnic = data['ethnic_long']
print(ds.quick_summary(ethnic))
ethnic = ethnic[~ethnic["variable"].str.contains("total", case=False, na=False)]
ethnic = ethnic[~ethnic["geography"].str.contains("toronto", case=False, na=False)]

ethnic = ethnic[
    ~ethnic["variable"].str.contains(
        r"visible minority|multiple visible minorities|origins|n\.o\.s\.|n\.i\.e\.",
        case=False,
        na=False,
        regex=True
    )
].reset_index(drop=True)

            dtype  col_index  missing_pct  n_unique
variable   object          0          0.0       382
geography  object          1          0.0        26
value       int64          2          0.0      1926
year        int64          3          0.0         2
category   object          4          0.0         1


In [24]:
ethnic['variable'].unique()

array(['Canadian', 'English', 'Irish', 'Scottish', 'French, n.o.s.',
       'German', 'Chinese', 'Italian', 'Indian (India)', 'Ukrainian',
       'Dutch', 'Polish', 'Québécois', 'British Isles, n.o.s.',
       'Filipino', 'French Canadian', 'Caucasian (White), n.o.s.',
       'First Nations (North American Indian), n.o.s.', 'Métis',
       'European, n.o.s.', 'Russian', 'Norwegian', 'Welsh', 'Portuguese',
       'American', 'Spanish', 'Swedish', 'Hungarian', 'Acadian',
       'Pakistani', 'African, n.o.s.', 'Jewish', 'Punjabi', 'Vietnamese',
       'Arab, n.o.s.', 'Greek', 'Jamaican', 'Asian, n.o.s.',
       'Cree, n.o.s.', 'Korean', 'Romanian', 'Lebanese', 'Iranian',
       'Christian, n.i.e.', 'Danish', 'North American Indigenous, n.o.s.',
       'Sikh', 'Austrian', 'Belgian', 'Haitian', 'Hindu', 'Mexican',
       'Mennonite', 'Swiss', 'Finnish', 'Sri Lankan', 'Croatian',
       'Japanese', 'South Asian, n.o.s.', "Mi'kmaq, n.o.s.",
       'Northern European, n.o.s.', 'Muslim', 'Egypt

### Mapping and transform

In [129]:
def map_ethnicity(variable: str) -> str | None:
    v = str(variable).strip().lower()

    # Indigenous
    if any(x in v for x in [
        "first nations", "north american indian", "métis", "metis", "inuit",
        "cree", "ojib", "algonquin", "anishinaabe", "iroquois", "haudenosaunee",
        "dene", "blackfoot", "mohawk", "maliseet", "mi'kmaq", "mikmaq",
        "huron", "wendat", "innu", "montagnais", "oji-cree", "saulteaux",
        "abenaki", "atikamekw", "cherokee", "maya", "mayan", "qalipu"
    ]):
        return "indigenous"

    # South Asian
    if any(x in v for x in [
        "indian", "east indian", "pakistani", "punjabi", "tamil", "bengali",
        "bangladeshi", "sri lankan", "gujarati", "nepali", "goan",
        "kashmiri", "telugu", "sinhalese", "malayali", "jatt", "south asian"
    ]):
        return "south_asian"

    # East / Southeast Asian
    if any(x in v for x in [
        "chinese", "filipino", "vietnamese", "korean", "japanese", "thai",
        "cambodian", "khmer", "laotian", "indonesian", "malaysian", "taiwanese",
        "hong konger", "singaporean", "burmese", "mongolian", "tibetan",
        "hmong", "igorot", "ilocano", "east and southeast asian",
        "east or southeast asian", "southeast asian"
    ]):
        return "east_se_asian"

    # Black / African / Caribbean
    if any(x in v for x in [
        "african", "black", "jamaican", "haitian", "nigerian", "somali",
        "ethiopian", "ghanaian", "cameroonian", "congolese", "sudanese",
        "rwandan", "burundian", "kenyan", "ugandan", "tanzanian", "zimbabwean",
        "ivorian", "senegalese", "guinean", "beninese", "libyan", "yoruba",
        "igbo", "ibo", "akan", "ashanti", "ewe", "fulani", "wolof", "oromo",
        "amhara", "bantu", "tigrinya", "tigrian", "dinka", "harari", "edo",
        "african caribbean", "african canadian", "african american",
        "african nova scotian", "caribbean", "west indian", "barbadian",
        "grenadian", "vincentian", "vincentian/grenadinian", "st. lucian",
        "trinidadian", "tobagonian", "indo-caribbean", "indo-guyanese"
    ]):
        return "black"

    # Middle Eastern / West Asian / Arab
    if any(x in v for x in [
        "arab", "iranian", "persian", "lebanese", "afghan", "turkish",
        "iraqi", "palestinian", "syrian", "egyptian", "algerian", "moroccan",
        "tunisian", "yemeni", "jordanian", "israeli", "armenian", "assyrian",
        "chaldean", "kurd", "kazakh", "kyrgyz", "uzbek", "uighur", "turkmen",
        "azerbaijani", "tajik", "georgian", "hazara", "saudi", "kuwaiti",
        "west asian", "west central asian", "middle eastern",
        "west or central asian or middle eastern"
    ]):
        return "middle_eastern"

    # Latin American
    if any(x in v for x in [
        "mexican", "brazilian", "colombian", "peruvian", "chilean", "cuban",
        "venezuelan", "ecuadorian", "argentinian", "guatemalan", "salvadorean",
        "dominican", "nicaraguan", "honduran", "paraguayan", "uruguayan",
        "costa rican", "panamanian", "bolivian", "belizean", "puerto rican",
        "latin american", "latin, central or south american",
        "latin, central and south american", "hispanic"
    ]):
        return "latin_american"

    # European / White
    if any(x in v for x in [
        "canadian", "québécois", "quebecois", "french canadian", "acadian",
        "english", "irish", "scottish", "welsh", "british", "british isles",
        "northern irish", "channel islander", "cornish", "manx", "celtic",
        "french", "alsatian", "corsican", "gaspesian", "norman",
        "german", "bavarian", "dutch", "flemish", "belgian", "luxembourger",
        "swiss", "austrian", "finnish", "swedish", "norwegian", "danish",
        "icelandic", "frisian", "northern european", "western european",
        "european", "caucasian", "white",
        "italian", "sicilian", "maltese", "portuguese", "spanish", "catalan",
        "greek", "cypriot", "albanian", "macedonian", "slovenian", "croatian",
        "serbian", "bosnian", "montenegrin", "kosovar", "romanian", "hungarian",
        "czech", "slovak", "polish", "ukrainian", "russian", "lithuanian",
        "latvian", "estonian", "bulgarian", "moldovan", "byelorussian",
        "slavic", "yugoslavian", "czechoslovakian", "eastern european",
        "southern european", "roma", "gypsy", "basque", "breton",
        "newfoundlander", "nova scotian", "new brunswicker", "ontarian",
        "albertan", "saskatchewanian", "manitoban", "british columbian",
        "prince edward islander", "cape bretoner", "franco ontarian",
        "north american, n.o.s.", "other north american origins"
    ]):
        return "european"

    # Oceania
    if any(x in v for x in [
        "australian", "new zealander", "fijian", "hawaiian", "maori",
        "samoan", "polynesian", "pacific islands", "oceania"
    ]):
        return "other"

    return "other"


def is_ethnicity_aggregate(variable: str) -> bool:
    v = str(variable).strip().lower()

    aggregate_patterns = [
        "total -",
        "visible minority",
        "multiple visible minorities",
        "origins",
        "n.o.s.",
        "n.i.e.",
    ]

    return any(p in v for p in aggregate_patterns)


def transform_ethnicity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["variable"] = df["variable"].astype(str).str.strip()

    # Drop aggregate / rollup / alternate-taxonomy rows before mapping
    df = df[~df["variable"].apply(is_ethnicity_aggregate)].copy()

    df["ethnicity_group"] = df["variable"].apply(map_ethnicity)

    grouped = (
        df.groupby(["geography", "year", "ethnicity_group"], as_index=False)["value"]
        .sum()
    )

    grouped["total"] = grouped.groupby(["geography", "year"])["value"].transform("sum")
    grouped["pct"] = grouped["value"] / grouped["total"]

    wide = (
        grouped.pivot(index=["geography", "year"], columns="ethnicity_group", values="pct")
        .reset_index()
        .rename(columns={
            "indigenous": "pct_indigenous",
            "european": "pct_european",
            "south_asian": "pct_south_asian",
            "east_se_asian": "pct_east_se_asian",
            "black": "pct_black",
            "middle_eastern": "pct_middle_eastern",
            "latin_american": "pct_latin_american",
            "other": "pct_other",
        })
    )

    wide.columns.name = None
    return wide


def check_unmapped_ethnicity_variables(df: pd.DataFrame) -> pd.DataFrame:
    check = df.copy()
    check["variable"] = check["variable"].astype(str).str.strip()
    check = check[~check["variable"].apply(is_ethnicity_aggregate)].copy()
    check["ethnicity_group"] = check["variable"].apply(map_ethnicity)

    return (
        check.groupby(["variable", "ethnicity_group"], as_index=False)
        .agg(rows=("variable", "size"), total_value=("value", "sum"))
        .sort_values(["ethnicity_group", "total_value", "variable"], ascending=[True, False, True])
    )
def check_missing_output_columns(wide_df: pd.DataFrame) -> list[str]:
    expected = [
        "pct_indigenous",
        "pct_european",
        "pct_south_asian",
        "pct_east_se_asian",
        "pct_black",
        "pct_middle_eastern",
        "pct_latin_american",
        "pct_other",
    ]
    return [c for c in expected if c not in wide_df.columns]

In [130]:
ethnicity_wide = transform_ethnicity(ethnic)

mapping_check = check_unmapped_ethnicity_variables(ethnic)
missing_cols = check_missing_output_columns(ethnicity_wide)

print(mapping_check.head(20))
print("Missing output columns:", missing_cols)

                   variable ethnicity_group  rows  total_value
43                    Black           black    50       504850
143                Jamaican           black    50       152095
247                  Somali           black    50        40780
269  Trinidadian/Tobagonian           black    50        37575
168              Macedonian           black    50        24920
90                Ethiopian           black    50        24165
200                Nigerian           black    50        20585
107                Ghanaian           black    50        17505
33                Barbadian           black    50        15370
6         African Caribbean           black    25        15290
110               Grenadian           black    50        12425
248           South African           black    50        10940
289                  Yoruba           black    50         6520
116                 Haitian           black    50         6240
283  Vincentian/Grenadinian           black    25      

In [132]:
ethnicity_wide[ethnicity_wide['geography']=='Ward 1']

,geography,year,pct_black,pct_east_se_asian,pct_european,pct_indigenous,pct_latin_american,pct_middle_eastern,pct_other,pct_south_asian
0,Ward 1,2016,0.235017,0.086725,0.221273,0.004193,0.051568,0.068429,0.021008,0.311787
1,Ward 1,2021,0.239226,0.099762,0.178947,0.001672,0.040792,0.089389,0.049615,0.300596


## Family Transformation 4/8

families_long

In [176]:
fam = data['families_long']
print(ds.quick_summary(fam))
fam[fam['geography']=='Ward 1']
fam.variable.unique()

             dtype  col_index  missing_pct  n_unique
variable    object          0          0.0        46
geography   object          1          0.0        26
value      float64          2          0.0      1792
year         int64          3          0.0         3
category    object          4          0.0         1


array(['Total children in census families in private households',
       'Under 6 years of age', '6 to 14 years', '15 to 17 years',
       '18 to 24 years', '25 years and over',
       'Total number of census families in private households - 25% sample data',
       'Total couple families', 'Married couples',
       'With children at home', 'Without children at home',
       'Common-law couples', 'Total one-parent families',
       'in which the parent is a woman+', 'in which the parent is a man+',
       'Total - Household type - 25% sample data',
       'One-census-family households without additional persons',
       'Couple-family households', 'One-parent-family households',
       'Multigenerational households',
       'Multiple-census-family households',
       'One-census-family households with additional persons',
       'Two-or-more-person non-census-family households', '1 person',
       'Total - Private households by household size - 25% sample data',
       '2 persons', '3 

In [177]:
fam = fam[~fam["geography"].str.contains("toronto", case=False, na=False)].reset_index(drop=True) # drop geography = Toronto

In [178]:
fam['variable'].unique()

array(['Total children in census families in private households',
       'Under 6 years of age', '6 to 14 years', '15 to 17 years',
       '18 to 24 years', '25 years and over',
       'Total number of census families in private households - 25% sample data',
       'Total couple families', 'Married couples',
       'With children at home', 'Without children at home',
       'Common-law couples', 'Total one-parent families',
       'in which the parent is a woman+', 'in which the parent is a man+',
       'Total - Household type - 25% sample data',
       'One-census-family households without additional persons',
       'Couple-family households', 'One-parent-family households',
       'Multigenerational households',
       'Multiple-census-family households',
       'One-census-family households with additional persons',
       'Two-or-more-person non-census-family households', '1 person',
       'Total - Private households by household size - 25% sample data',
       '2 persons', '3 

In [195]:
fam2021= fam[(fam["geography"] == "Ward 1") & (fam["year"] == 2021)]
fam2016= fam[(fam["geography"] == "Ward 1") & (fam["year"] == 2016)]

In [211]:
fam2016.variable.unique()

array(['Total - Couple census families in private households - 25% sample data',
       'Couples without children', 'Couples with children', '1 child',
       '2 children', '3 or more children',
       'Total - Lone-parent census families in private households - 25% sample data',
       'Total children in census families in private households',
       'Under 6 years of age', '6 to 14 years', '15 to 17 years',
       '18 to 24 years', '25 years and over',
       'Total - Private households by household type - 25% sample data',
       'One-census-family households',
       'Multiple-census-family households',
       'Non-census-family households',
       'Total - Private households by household size - 25% sample data',
       '1 person', '2 persons', '3 persons', '4 persons',
       '5 or more persons', 'Number of persons in private households',
       'Average household size'], dtype=object)

### Mapping, Transform, and backfill

for 2016, it was manually calculated the number of households with children and without children through raw count numbers of # of childrens iin census data

In [219]:
def transform_family_households(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["variable"] = df["variable"].astype(str).str.strip()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    child_age_map = {
        "Under 6 years of age": "children_under_6",
        "6 to 14 years": "children_6_14",
        "15 to 17 years": "children_15_17",
        "18 to 24 years": "children_18_24",
        "25 years and over": "children_25_plus",
    }

    child_count_map = {
        "1 child": "1_child",
        "2 children": "2_children",
        "3 or more children": "3plus_children",
    }

    def compute_pct(subset: pd.DataFrame, group_col: str) -> pd.DataFrame:
        grouped = (
            subset.groupby(["geography", "year", group_col], as_index=False)["value"]
            .sum()
        )
        grouped["denominator"] = grouped.groupby(["geography", "year"])["value"].transform("sum")
        grouped["pct"] = grouped["value"] / grouped["denominator"]

        return (
            grouped.pivot(index=["geography", "year"], columns=group_col, values="pct")
            .reset_index()
            .rename_axis(None, axis=1)
        )

    # children age distribution
    age_df = df[df["variable"].isin(child_age_map)].copy()
    age_df["child_age_group"] = age_df["variable"].map(child_age_map)
    age_wide = compute_pct(age_df, "child_age_group")

    # children count distribution among couple families with children
    count_df = df[df["variable"].isin(child_count_map)].copy()
    count_df["child_count_group"] = count_df["variable"].map(child_count_map)
    count_wide = compute_pct(count_df, "child_count_group")

    # pct_with_children, using year-specific available schema
    couple_2016 = df[df["variable"].isin([
        "Couples without children",
        "Couples with children",
    ])].copy()
    
    couple_2016_wide = (
        couple_2016.pivot_table(
            index=["geography", "year"],
            columns="variable",
            values="value",
            aggfunc="sum"
        )
        .reset_index()
    )
    
    if {"Couples with children", "Couples without children"}.issubset(couple_2016_wide.columns):
        couple_2016_wide["pct_with_children"] = (
            couple_2016_wide["Couples with children"] /
            (
                couple_2016_wide["Couples with children"]
                + couple_2016_wide["Couples without children"]
            )
        )
        couple_2016_wide = couple_2016_wide[["geography", "year", "pct_with_children"]]
    else:
        couple_2016_wide = pd.DataFrame(columns=["geography", "year", "pct_with_children"])
    
    
    couple_2021 = df[df["variable"].isin([
        "With children at home",
        "Without children at home",
    ])].copy()
    
    couple_2021_wide = (
        couple_2021.pivot_table(
            index=["geography", "year"],
            columns="variable",
            values="value",
            aggfunc="sum"
        )
        .reset_index()
    )
    
    if {"With children at home", "Without children at home"}.issubset(couple_2021_wide.columns):
        couple_2021_wide["pct_with_children"] = (
            couple_2021_wide["With children at home"] /
            (
                couple_2021_wide["With children at home"]
                + couple_2021_wide["Without children at home"]
            )
        )
        couple_2021_wide = couple_2021_wide[["geography", "year", "pct_with_children"]]
    else:
        couple_2021_wide = pd.DataFrame(columns=["geography", "year", "pct_with_children"])
    
    
    couple_wide = pd.concat(
        [couple_2016_wide, couple_2021_wide],
        ignore_index=True
    ).drop_duplicates(
        subset=["geography", "year"],
        keep="first"
    )

    couple_wide = couple_wide[["geography", "year", "pct_with_children"]]

    final = (
        age_wide
        .merge(count_wide, on=["geography", "year"], how="outer")
        .merge(couple_wide, on=["geography", "year"], how="outer")
    )

    final = final.rename(columns={
        "children_under_6": "pct_children_under_6",
        "children_6_14": "pct_children_6_14",
        "children_15_17": "pct_children_15_17",
        "children_18_24": "pct_children_18_24",
        "children_25_plus": "pct_children_25_plus",
        "1_child": "pct_1_child",
        "2_children": "pct_2_children",
        "3plus_children": "pct_3plus_children",
    })

    pct_cols = [c for c in final.columns if c.startswith("pct_")]
    final[pct_cols] = final[pct_cols].fillna(pd.NA)
    # VALIDATION
    age_ok = final[[
        "pct_children_under_6",
        "pct_children_6_14",
        "pct_children_15_17",
        "pct_children_18_24",
        "pct_children_25_plus",
    ]].sum(axis=1).between(0.99, 1.01)
    
    count_ok = final[[
        "pct_1_child",
        "pct_2_children",
        "pct_3plus_children",
    ]].sum(axis=1).between(0.99, 1.01) | final["year"].ne(2016)
    
    pct_ok = final["pct_with_children"].between(0, 1) | final["pct_with_children"].isna()
    
    if not (age_ok & count_ok & pct_ok).all():
        raise ValueError("Family validation failed")
    return final




In [220]:
family_wide = transform_family_households(fam)
#validation = validate_family_households_output(family_wide)
#unmapped = check_unmapped_family_variables(fam)

In [221]:
family_wide[family_wide['year']==2021].head()

,geography,year,pct_children_15_17,pct_children_18_24,pct_children_25_plus,pct_children_6_14,pct_children_under_6,pct_1_child,pct_2_children,pct_3plus_children,pct_with_children
2,Ward 1,2021,0.097406,0.216038,0.253656,0.267689,0.165212,NaN,NaN,NaN,0.645853
5,Ward 10,2021,0.066980,0.143975,0.210955,0.267606,0.310485,NaN,NaN,NaN,0.258479
8,Ward 11,2021,0.093794,0.199490,0.198073,0.300085,0.208558,NaN,NaN,NaN,0.369798
11,Ward 12,2021,0.106669,0.194456,0.158899,0.320209,0.219767,NaN,NaN,NaN,0.426980
14,Ward 13,2021,0.088135,0.196520,0.188534,0.276098,0.250713,NaN,NaN,NaN,0.314282


## Households transformations 5/8
households_long

In [237]:
house = data['households_long']
print(ds.quick_summary(house))
#house = drop_totals(house)

             dtype  col_index  missing_pct  n_unique
variable    object          0          0.0        38
geography   object          1          0.0        26
value      float64          2          0.0       882
year         int64          3          0.0         1
category    object          4          0.0         1


In [243]:
house.year.unique()

array([2011])

In [261]:
house[house['geography']=='Ward 1']

,variable,geography,value,year,category
0,Total number of occupied private dwellings by ...,Ward 1,37085.0,2011,households
1,Single-detached house,Ward 1,11580.0,2011,households
2,Semi-detached house,Ward 1,1330.0,2011,households
3,Row house,Ward 1,3620.0,2011,households
4,"Apartment, duplex",Ward 1,3475.0,2011,households
5,"Apartment, building that has five or more storeys",Ward 1,15215.0,2011,households
6,"Apartment, building that has fewer than five s...",Ward 1,1860.0,2011,households
7,Other single-attached house,Ward 1,5.0,2011,households
8,Movable dwelling,Ward 1,0.0,2011,households
9,Total number of private households by househol...,Ward 1,37085.0,2011,households


In [264]:
house['variable'].unique()

array(['Total number of occupied private dwellings by structural type of dwelling',
       'Single-detached house', 'Semi-detached house', 'Row house',
       'Apartment, duplex',
       'Apartment, building that has five or more storeys',
       'Apartment, building that has fewer than five storeys',
       'Other single-attached house', 'Movable dwelling',
       'Total number of private households by household type',
       'Census family households', 'One-family only households',
       'Couple family households', 'Without children', 'With children',
       'Lone-parent family households', 'Other family households',
       'One-family households with persons not in a census family',
       'Two-or-more-family households', 'Non-census family households',
       'One-person households', 'Two-or-more-person households',
       'Total number of private households by household size', '1 person',
       '2 persons', '3 persons', '4 persons', '5 persons',
       '6 or more persons',
     

In [245]:
house = house[~house["geography"].str.contains("Toronto", case=False, na=False)].reset_index(drop=True)

## Mapping, transform

In [262]:
def transform_households_wide(households_long: pd.DataFrame) -> pd.DataFrame:
    df = households_long.copy()
    raw = households_long.copy()

    # clean both
    for d in (df, raw):
        d["variable"] = d["variable"].astype(str).str.strip()
        d["value"] = pd.to_numeric(d["value"], errors="coerce")

    # ----------------------------
    # FILTER (features + totals)
    # ----------------------------
    df = df[
        df["variable"].isin(HOUSEHOLDS_FEATURE_MAP)
        | df["variable"].str.contains("Total|Number of", case=False, na=False)
    ].copy()

    df["feature"] = df["variable"].map(HOUSEHOLDS_FEATURE_MAP)
    df["denominator_variable"] = df["variable"].map(HOUSEHOLDS_DENOMINATOR_MAP)

    # ----------------------------
    # DENOMINATORS
    # ----------------------------
    denom = (
        df.groupby(["geography", "year", "denominator_variable"], as_index=False)["value"]
        .sum()
        .rename(columns={"value": "denominator_value"})
    )

    df = df.merge(
        denom,
        on=["geography", "year", "denominator_variable"],
        how="left"
    )

    df["pct_value"] = df["value"] / df["denominator_value"]

    # ----------------------------
    # BASE FEATURES
    # ----------------------------
    wide = (
        df.groupby(["geography", "year", "feature"], as_index=False)["pct_value"]
        .sum()
        .pivot_table(
            index=["geography", "year"],
            columns="feature",
            values="pct_value",
            aggfunc="sum"
        )
        .reset_index()
    )

    wide.columns.name = None

    # ----------------------------
    # CHILDREN (MAIN BLOCK ONLY)
    # ----------------------------
    fam = raw.sort_values(["geography", "year"]).reset_index(drop=True)

    mask = fam["variable"] == "Census family households"

    blocks = []
    for i in fam.index[mask]:
        blocks.append(fam.loc[i:i+5])  # fixed structure

    fam_block = pd.concat(blocks)

    fam_block = fam_block[fam_block["variable"].isin([
        "Census family households",
        "Couple family households",
        "With children",
        "Without children",
        "Lone-parent family households",
    ])]

    fam_wide = (
        fam_block.pivot_table(
            index=["geography", "year"],
            columns="variable",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    # safe computation
    if all(col in fam_wide.columns for col in [
        "Census family households",
        "With children",
        "Without children",
        "Lone-parent family households",
    ]):
        fam_wide["pct_with_children"] = (
            fam_wide["With children"]
            + fam_wide["Lone-parent family households"]
        ) / fam_wide["Census family households"]

        fam_wide["pct_without_children"] = (
            fam_wide["Without children"]
        ) / fam_wide["Census family households"]

        fam_wide["pct_single_parent"] = (
            fam_wide["Lone-parent family households"]
            / (
                fam_wide["With children"]
                + fam_wide["Lone-parent family households"]
            )
        )

        fam_wide = fam_wide[[
            "geography",
            "year",
            "pct_with_children",
            "pct_without_children",
            "pct_single_parent",
        ]]

        wide = wide.merge(fam_wide, on=["geography", "year"], how="left")

    # ----------------------------
    # FINAL ORDER
    # ----------------------------
    ordered_cols = [
        "geography",
        "year",

        "pct_1_person",
        "pct_2_person",
        "pct_3_person",
        "pct_4_person",
        "pct_5plus_person",

        "pct_with_children",
        "pct_without_children",
        "pct_single_parent",

        "pct_multi_family",
        "pct_non_family",

        "pct_detached",
        "pct_attached/other",
        "pct_apartment",
        "pct_condo",
        "pct_non_condo",
    ]

    for col in ordered_cols:
        if col not in wide.columns:
            wide[col] = pd.NA

    return wide[ordered_cols]

In [263]:
house_trans = transform_households_wide(house)
house_trans

,geography,year,pct_1_person,pct_2_person,pct_3_person,pct_4_person,pct_5plus_person,pct_with_children,pct_without_children,pct_single_parent,pct_multi_family,pct_non_family,pct_detached,pct_attached/other,pct_apartment,pct_condo,pct_non_condo
0,Ward 1,2011,0.183338,0.241979,0.189809,0.192100,0.192774,0.604378,0.185814,0.316720,0.233737,0.766263,0.312256,0.227316,0.460429,0.228106,0.771894
1,Ward 10,2011,0.519724,0.337646,0.083457,0.036267,0.022906,0.345638,0.576063,0.384304,0.015144,0.984856,0.012301,0.088229,0.899470,0.718432,0.281568
2,Ward 11,2011,0.440247,0.329950,0.111030,0.075555,0.043219,0.427628,0.477133,0.282800,0.020457,0.979543,0.091765,0.167312,0.740923,0.255308,0.744692
3,Ward 12,2011,0.467088,0.304918,0.107444,0.079315,0.041236,0.479162,0.439114,0.296988,0.014079,0.985921,0.174146,0.103435,0.722419,0.360172,0.639828
4,Ward 13,2011,0.548737,0.302525,0.081848,0.041639,0.025250,0.428728,0.501778,0.425016,0.008263,0.991737,0.005050,0.064310,0.930640,0.175074,0.824926
5,Ward 14,2011,0.350218,0.309383,0.155305,0.121611,0.063483,0.545644,0.339723,0.301565,0.045983,0.954017,0.195604,0.333742,0.470654,0.053412,0.946588
6,Ward 15,2011,0.304237,0.280373,0.150204,0.162966,0.102221,0.617545,0.302866,0.195359,0.040575,0.959425,0.381876,0.117294,0.500830,0.112967,0.887033
7,Ward 16,2011,0.303550,0.300359,0.171254,0.137748,0.087089,0.608394,0.283619,0.320161,0.063754,0.936246,0.149541,0.142230,0.708228,0.288070,0.711930
8,Ward 17,2011,0.221390,0.295756,0.216397,0.171725,0.094731,0.591176,0.272371,0.239844,0.119132,0.880868,0.240999,0.229435,0.529566,0.300092,0.699908
9,Ward 18,2011,0.300966,0.315888,0.180492,0.137371,0.065284,0.552921,0.329038,0.251088,0.062642,0.937358,0.278284,0.083397,0.638319,0.513543,0.486457


In [247]:
house['variable'].value_counts()
house

,variable,geography,value,year,category
0,Total number of occupied private dwellings by ...,Ward 1,37085.0,2011,households
1,Single-detached house,Ward 1,11580.0,2011,households
2,Semi-detached house,Ward 1,1330.0,2011,households
3,Row house,Ward 1,3620.0,2011,households
4,"Apartment, duplex",Ward 1,3475.0,2011,households
...,...,...,...,...,...
1045,Owned - Part of a condominium,Ward 25,3645.0,2011,households
1046,Owned - Not part of a condominium,Ward 25,22045.0,2011,households
1047,Rented,Ward 25,5940.0,2011,households
1048,Rented - Part of a condominium,Ward 25,800.0,2011,households


### TO DO, UNION WITH DWELLINGS AND FAMILY

## Income Transformations 6/8

In [100]:
income = data['income_long']
print(ds.quick_summary(income))
income = drop_totals(income)

            dtype  col_index  missing_pct  n_unique
variable   object          0     0.000000        89
geography  object          1     0.000000        26
value      object          2     0.032864      2753
year        int64          3     0.000000         2
category   object          4     0.000000         1


In [104]:
income['variable'].unique()

array(['Total - Household total income groups in 2020 for private households - 25% sample data',
       'Under $5,000', '$5,000 to $9,999', '$10,000 to $14,999',
       '$15,000 to $19,999', '$20,000 to $24,999', '$25,000 to $29,999',
       '$30,000 to $34,999', '$35,000 to $39,999', '$40,000 to $44,999',
       '$45,000 to $49,999', '$50,000 to $59,999', '$60,000 to $69,999',
       '$70,000 to $79,999', '$80,000 to $89,999', '$90,000 to $99,999',
       '$100,000 and over', '$100,000 to $124,999',
       '$125,000 to $149,999', '$150,000 to $199,999',
       '$200,000 and over',
       'Total - Income statistics in 2020 for private households by household size - 25% sample data',
       'Average total income of households in 2020 ($)',
       'Median total income of households in 2020 ($)',
       'Total - Income statistics in 2020 for one-person private households - 25% sample data',
       'Average total income of one-person households in 2020 ($)',
       'Median total income of 

### Mapping, Transform, melt

In [106]:
import pandas as pd


INCOME_FEATURE_MAP = {
    # affordability
    "Average monthly shelter costs for rented dwellings ($)": "avg_rent_cost",
    "% of tenant households spending 30% or more of its income on shelter costs": "pct_renters_30_plus_income_on_shelter",
    "Average monthly shelter costs for owned dwellings ($)": "avg_owner_cost",
    "% of owner households spending 30% or more of its income on shelter costs": "pct_owners_30_plus_income_on_shelter",

    # poverty / low income
    "Prevalence of low income based on the Low-income measure, after tax (LIM-AT) (%)": "pct_low_income_lim_at",

    # income composition
    "Employment income %": "pct_employment_income",
    "Government transfers %": "pct_government_transfers",
}


INCOME_BUCKET_MAP = {
    # low income: under 30k
    "Under $5,000": "pct_low_income",
    "$5,000 to $9,999": "pct_low_income",
    "$10,000 to $14,999": "pct_low_income",
    "$15,000 to $19,999": "pct_low_income",
    "$20,000 to $24,999": "pct_low_income",
    "$25,000 to $29,999": "pct_low_income",

    # lower-middle: 30k to 60k
    "$30,000 to $34,999": "pct_lower_mid_income",
    "$35,000 to $39,999": "pct_lower_mid_income",
    "$40,000 to $44,999": "pct_lower_mid_income",
    "$45,000 to $49,999": "pct_lower_mid_income",
    "$50,000 to $59,999": "pct_lower_mid_income",

    # middle: 60k to 100k
    "$60,000 to $69,999": "pct_mid_income",
    "$70,000 to $79,999": "pct_mid_income",
    "$80,000 to $89,999": "pct_mid_income",
    "$90,000 to $99,999": "pct_mid_income",

    # high: 100k+
    "$100,000 to $124,999": "pct_high_income",
    "$125,000 to $149,999": "pct_high_income",
    "$150,000 to $199,999": "pct_high_income",

    # very high: 150k+
    "$200,000 and over": "pct_very_high_income",
}


def transform_income_wide(income_long: pd.DataFrame) -> pd.DataFrame:
    """
    Transform income_long into a wide feature table consistent with:
    - family_wide
    - dwellings_wide
    - households_wide

    Expected input columns:
    - geography
    - year
    - variable
    - value

    Output:
    - geography
    - year
    - selected pct_* and avg_* income features
    """

    df = income_long.copy()

    df["variable"] = df["variable"].astype(str).str.strip()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # percentage/dollar features already provided directly
    direct = df[df["variable"].isin(INCOME_FEATURE_MAP)].copy()
    direct["feature"] = direct["variable"].map(INCOME_FEATURE_MAP)

    # convert % fields from whole-number percent to decimal
    pct_direct_features = [
        "pct_renters_30_plus_income_on_shelter",
        "pct_owners_30_plus_income_on_shelter",
        "pct_low_income_lim_at",
        "pct_employment_income",
        "pct_government_transfers",
    ]

    direct.loc[
        direct["feature"].isin(pct_direct_features),
        "value"
    ] = direct.loc[
        direct["feature"].isin(pct_direct_features),
        "value"
    ] / 100

    direct_wide = (
        direct.pivot_table(
            index=["geography", "year"],
            columns="feature",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    # income distribution buckets
    buckets = df[df["variable"].isin(INCOME_BUCKET_MAP)].copy()
    buckets["feature"] = buckets["variable"].map(INCOME_BUCKET_MAP)

    bucket_wide = (
        buckets.groupby(["geography", "year", "feature"], as_index=False)["value"]
        .sum()
        .pivot_table(
            index=["geography", "year"],
            columns="feature",
            values="value",
            aggfunc="sum"
        )
        .reset_index()
    )

    # If bucket values are counts, convert to percentages.
    # If they are already percentages, remove this block.
    bucket_cols = [
        "pct_low_income",
        "pct_lower_mid_income",
        "pct_mid_income",
        "pct_high_income",
        "pct_very_high_income",
    ]

    available_bucket_cols = [c for c in bucket_cols if c in bucket_wide.columns]

    if available_bucket_cols:
        bucket_total = bucket_wide[
            ["pct_low_income", "pct_lower_mid_income", "pct_mid_income", "pct_high_income"]
        ].sum(axis=1)

        for col in available_bucket_cols:
            bucket_wide[col] = bucket_wide[col] / bucket_total

    income_wide = direct_wide.merge(
        bucket_wide,
        on=["geography", "year"],
        how="outer"
    )

    income_wide.columns.name = None

    ordered_cols = [
        "geography",
        "year",

        # income distribution
        "pct_low_income",
        "pct_lower_mid_income",
        "pct_mid_income",
        "pct_high_income",
        "pct_very_high_income",

        # affordability
        "pct_renters_30_plus_income_on_shelter",
        "pct_owners_30_plus_income_on_shelter",
        "avg_rent_cost",
        "avg_owner_cost",

        # poverty
        "pct_low_income_lim_at",

        # income composition
        "pct_employment_income",
        "pct_government_transfers",
    ]

    for col in ordered_cols:
        if col not in income_wide.columns:
            income_wide[col] = pd.NA

    return income_wide[ordered_cols]

In [45]:
income_wide = transform_income_wide(income)
ds.quick_summary(income_wide)
income_wide

,geography,year,pct_low_income,pct_lower_mid_income,pct_mid_income,pct_high_income,pct_very_high_income,pct_renters_30_plus_income_on_shelter,pct_owners_30_plus_income_on_shelter,avg_rent_cost,avg_owner_cost,pct_low_income_lim_at,pct_employment_income,pct_government_transfers
0,Ward 1,2016,0.122387,0.254377,0.328064,0.295173,0.050399,0.428,0.273,1064.0,1425.0,0.225,0.718,0.193
1,Ward 1,2021,0.061058,0.217239,0.338499,0.383204,0.089766,0.342,0.235,1328.0,1664.0,0.129,0.620,0.292
2,Ward 10,2016,0.100783,0.148810,0.336421,0.413986,0.088418,0.446,0.360,1673.0,1916.0,0.173,0.877,0.034
3,Ward 10,2021,0.073140,0.125073,0.313278,0.488509,0.114913,0.437,0.352,1988.0,2290.0,0.141,0.842,0.077
4,Ward 11,2016,0.143778,0.147109,0.262983,0.446131,0.125921,0.523,0.279,1560.0,2029.0,0.202,0.710,0.035
5,Ward 11,2021,0.093829,0.138123,0.271201,0.496846,0.142296,0.488,0.298,1976.0,2592.0,0.153,0.692,0.073
6,Ward 12,2016,0.116510,0.160466,0.292126,0.430898,0.116201,0.471,0.229,1384.0,2047.0,0.161,0.692,0.047
7,Ward 12,2021,0.076416,0.146727,0.293191,0.483666,0.130109,0.446,0.248,1768.0,2492.0,0.125,0.686,0.085
8,Ward 13,2016,0.208567,0.179245,0.296106,0.316082,0.067564,0.465,0.346,1214.0,1840.0,0.312,0.810,0.074
9,Ward 13,2021,0.141987,0.166305,0.305956,0.385751,0.085259,0.431,0.351,1520.0,2172.0,0.222,0.758,0.140


In [46]:
income_ready = income_wide.melt(
    id_vars=["geography", "year"],
    var_name="metric_name",
    value_name="value"
).assign(source_group="income")

In [105]:
income_ready[income_ready['geography']=='Ward 1']

,geography,year,metric_name,value,source_group
0,Ward 1,2016,pct_low_income,0.122387,income
1,Ward 1,2021,pct_low_income,0.061058,income
50,Ward 1,2016,pct_lower_mid_income,0.254377,income
51,Ward 1,2021,pct_lower_mid_income,0.217239,income
100,Ward 1,2016,pct_mid_income,0.328064,income
101,Ward 1,2021,pct_mid_income,0.338499,income
150,Ward 1,2016,pct_high_income,0.295173,income
151,Ward 1,2021,pct_high_income,0.383204,income
200,Ward 1,2016,pct_very_high_income,0.050399,income
201,Ward 1,2021,pct_very_high_income,0.089766,income


## Labour Transformations 7/8
labour_long

In [91]:
labour = data['labour_long']

In [85]:
labour

,variable,geography,value,year,category
0,Total - Population aged 15 years and over by L...,Toronto,2294785.0,2016,labour
1,In the labour force,Toronto,1483680.0,2016,labour
2,Employed,Toronto,1361375.0,2016,labour
3,Unemployed,Toronto,122305.0,2016,labour
4,Not in the labour force,Toronto,811110.0,2016,labour
...,...,...,...,...,...
1217,Total - Place of work status for the employed ...,Ward 25,48605.0,2016,labour
1218,Worked at home,Ward 25,2310.0,2016,labour
1219,Worked outside Canada,Ward 25,240.0,2016,labour
1220,No fixed workplace address,Ward 25,5360.0,2016,labour


In [86]:
labour_d = drop_totals(labour)
print(ds.quick_summary(labour))

             dtype  col_index  missing_pct  n_unique
variable    object          0          0.0        42
geography   object          1          0.0        25
value      float64          2          0.0       821
year         int64          3          0.0         1
category    object          4          0.0         1


In [88]:
labour_d[labour_d['value'].isnull()]

,variable,geography,value,year,category


In [89]:
#labour[labour['variable']]
labour_d['variable'].value_counts()

variable
Participation rate                                                               50
In the labour force                                                              25
61 Educational services                                                          25
44-45 Retail trade                                                               25
48-49 Transportation and warehousing                                             25
51 Information and cultural industries                                           25
52 Finance and insurance                                                         25
53 Real estate and rental and leasing                                            25
54 Professional, scientific and technical services                               25
55 Management of companies and enterprises                                       25
56 Administrative and support, waste management and remediation services         25
62 Health care and social assistance                               

In [92]:
a = [labour['variable'].isin(['In the labour force','Employed','Unemployed', 'Not in the labour force', 'Participation rate','Unemployment rate'])]
a = a[a['geography']=='Ward 1']

In [95]:
a

,variable,geography,value,year,category
48,In the labour force,Ward 1,57310.0,2016,labour
49,Employed,Ward 1,51215.0,2016,labour
50,Unemployed,Ward 1,6095.0,2016,labour
51,Not in the labour force,Ward 1,37985.0,2016,labour
52,Unemployment rate,Ward 1,10.6,2016,labour
54,Participation rate,Ward 1,65.2,2016,labour
56,Participation rate,Ward 1,55.3,2016,labour


In [94]:
labour[labour['geography']=='Ward 1']

,variable,geography,value,year,category
47,Total - Population aged 15 years and over by L...,Ward 1,95290.0,2016,labour
48,In the labour force,Ward 1,57310.0,2016,labour
49,Employed,Ward 1,51215.0,2016,labour
50,Unemployed,Ward 1,6095.0,2016,labour
51,Not in the labour force,Ward 1,37985.0,2016,labour
52,Unemployment rate,Ward 1,10.6,2016,labour
53,Total - Males aged 15 years and over by Labour...,Ward 1,NaN,2016,labour
54,Participation rate,Ward 1,65.2,2016,labour
55,Total - Females aged 15 years and over by Labo...,Ward 1,NaN,2016,labour
56,Participation rate,Ward 1,55.3,2016,labour
